# Student Zero-Shot and LoRA SFT Experiments

Use this notebook to run the next project stage: validate teacher data, evaluate the base student zero-shot, build SFT files, train one or more LoRA configs, and compare output metrics.

Recommended flow: run zero-shot first, inspect metrics and outputs, then enable one small SFT config before running a larger config.

## 1. Colab Pull / Local Setup

Run this first. In Colab it clones or pulls the GitHub repo and switches into the project directory. Locally it leaves your current checkout alone.

In [ ]:
import importlib.util
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/kdnehihi/strategy-distill-rl.git"
REPO_DIR = Path("/content/strategy-distill-rl")
IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    if REPO_DIR.exists():
        print(f"Pulling latest repo in {REPO_DIR}")
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", "main"], check=True)
    else:
        print(f"Cloning repo to {REPO_DIR}")
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
else:
    print(f"Local run detected. Current directory: {Path.cwd()}")

print(f"Working directory: {Path.cwd()}")


## 2. Install Dependencies

This cell checks the packages needed for SFT and installs only missing ones. It intentionally avoids reinstalling the full requirements file so Colab does not spend time on unrelated packages such as vLLM.

In [ ]:
AUTO_INSTALL_MISSING_DEPENDENCIES = True

REQUIRED_PACKAGES = {
    "datasets": "datasets",
    "peft": "peft",
    "accelerate": "accelerate",
    "transformers": "transformers",
    "pandas": "pandas",
    "tqdm": "tqdm",
}


def install_missing_dependencies():
    import importlib.util
    import sys

    missing = [
        package_name
        for import_name, package_name in REQUIRED_PACKAGES.items()
        if importlib.util.find_spec(import_name) is None
    ]
    if not missing:
        print("All SFT dependencies are already installed.")
        return

    if not AUTO_INSTALL_MISSING_DEPENDENCIES:
        raise ModuleNotFoundError(
            "Missing packages: " + ", ".join(missing) +
            ". Set AUTO_INSTALL_MISSING_DEPENDENCIES=True and rerun this cell."
        )

    print("Installing missing SFT dependencies:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)


def parse_version_tuple(version):
    parts = []
    for chunk in version.split(".")[:3]:
        try:
            parts.append(int(chunk))
        except ValueError:
            parts.append(0)
    while len(parts) < 3:
        parts.append(0)
    return tuple(parts)


def fix_incompatible_torchao():
    import importlib.metadata
    import sys

    try:
        version = importlib.metadata.version("torchao")
    except importlib.metadata.PackageNotFoundError:
        print("torchao is not installed; no compatibility fix needed.")
        return

    if parse_version_tuple(version) >= (0, 16, 0):
        print(f"torchao {version} is compatible.")
        return

    print(
        f"Found torchao {version}, which breaks PEFT LoRA injection. "
        "Uninstalling torchao because this SFT notebook does not need it."
    )
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)


install_missing_dependencies()
fix_incompatible_torchao()


## 3. Experiment Config

Change the flags and config list here. Keep the first run small; then increase samples or epochs once the output format looks stable.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()

# Full distillation run: scale the student from 1.5B to the public 7B Math model while keeping the same teacher traces.
MODEL_NAME = "Qwen/Qwen2.5-Math-7B-Instruct"
TEACHER_PATH = "data/gsm8k_teacher_preview_5000.jsonl"
EVAL_INPUT_PATH = "data/gsm8k_clean_test.jsonl"
SFT_TRAIN_PATH = "data/sft_strategy_train.jsonl"
SFT_VAL_PATH = "data/sft_strategy_val.jsonl"
SFT_TRAIN_SIZE = 4000  # Leaves the remaining usable teacher records for validation.

# From Google Drive: My Drive > RL > Data / Checkpoints
DRIVE_DATA_DIR = "/content/drive/MyDrive/RL/Data"
DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/RL/Checkpoints"
DRIVE_CHECKPOINT_ARCHIVE = "/content/drive/MyDrive/RL/Checkpoints/balanced_r16_a32_4000_20260630_233638.zip"
DRIVE_CHECKPOINT_NAME = "qwen7b_strategy_r16_a32_4000_e1"

RUN_VALIDATE_TEACHER = True
RUN_ZERO_SHOT = True
RUN_BUILD_SFT = True
RUN_RESTORE_DRIVE_CHECKPOINT = False
RUN_TRAINING = True
RUN_ADAPTER_EVAL = True
RUN_GENERATE_RL_ROLLOUTS = False  # Keep rollout generation off; RL notebooks reuse/rescore existing rollout files.
RUN_DOWNLOAD_RL_ROLLOUTS = False
RUN_SAVE_SFT_CHECKPOINT_TO_DRIVE = True

ZERO_SHOT_NUM_SAMPLES = -1
FINAL_EVAL_NUM_SAMPLES = -1
EVAL_BATCH_SIZE = 8
EVAL_MAX_NEW_TOKENS = 256

# RL rollout generation is intentionally off here. Use notebooks 04/07 to reuse and rescore existing rollout files.
ROLLOUT_INPUT_PATH = "data/gsm8k_clean_train.jsonl"
ROLLOUT_OUTPUT_PATH = "data/rl_rollouts_student.jsonl"
ROLLOUT_FORMAT_VALID_PATH = "data/rl_rollouts_student_format_valid.jsonl"
ROLLOUT_NUM_SAMPLES = -1
ROLLOUT_NUM_GENERATIONS = 4
ROLLOUT_BATCH_SIZE = 4
ROLLOUT_MAX_NEW_TOKENS = 512
ROLLOUT_TEMPERATURE = 0.7
ROLLOUT_TOP_P = 0.9

AUDIT_SAMPLE_COUNT = 50
# "auto" audits the first active adapter output if it exists, otherwise zero-shot.
AUDIT_RUN_NAME = "auto"

ACTIVE_TRAIN_CONFIG_NAMES = [DRIVE_CHECKPOINT_NAME]

TRAIN_CONFIGS = [
    {
        "name": "qwen7b_strategy_r16_a32_4000_e1",
        "max_train_samples": 100000,
        "max_val_samples": 100000,
        "epochs": 1.0,
        "lr": 2e-4,
        "lora_r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.05,
        "grad_accum": 8,
    },
    {
        "name": "qwen7b_strategy_r16_a32_4000_e2_lr1e4",
        "max_train_samples": 100000,
        "max_val_samples": 100000,
        "epochs": 2.0,
        "lr": 1e-4,
        "lora_r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.05,
        "grad_accum": 8,
    },
]

RUNS_DIR = Path("runs/student_sft")
CHECKPOINTS_DIR = Path("checkpoints/student_sft")
RUNS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)


## 4. Helpers

These helpers run repo scripts and load metric JSON files into a comparison table.

In [ ]:
import json
import subprocess
from pathlib import Path

import pandas as pd


def run_command(args):
    command = [str(arg) for arg in args]
    print("$", " ".join(command))
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    output_lines = []
    for line in process.stdout:
        print(line, end="")
        output_lines.append(line)
    return_code = process.wait()
    if return_code != 0:
        tail = "".join(output_lines[-80:])
        raise RuntimeError(
            f"Command failed with exit code {return_code}: {' '.join(command)}\n"
            f"Last output lines:\n{tail}"
        )


def load_json(path):
    with Path(path).open("r", encoding="utf-8") as f:
        return json.load(f)


def read_jsonl(path, limit=None):
    rows = []
    with Path(path).open("r", encoding="utf-8") as f:
        for line in f:
            if limit is not None and len(rows) >= limit:
                break
            rows.append(json.loads(line))
    return rows


def metric_row(name, metrics_path, train_metrics_path=None):
    metrics = load_json(metrics_path)
    row = {
        "run": name,
        "total": metrics.get("total"),
        "accuracy": metrics.get("accuracy"),
        "loose_math_accuracy": metrics.get("loose_math_accuracy"),
        "format_valid_rate": metrics.get("format_valid_rate"),
        "usable_rate": metrics.get("usable_rate"),
        "correct": metrics.get("correct"),
        "loose_correct": metrics.get("loose_correct"),
        "format_valid": metrics.get("format_valid"),
        "usable": metrics.get("usable"),
        "metrics_path": str(metrics_path),
    }
    if train_metrics_path and Path(train_metrics_path).exists():
        train_metrics = load_json(train_metrics_path)
        row["eval_loss"] = train_metrics.get("eval", {}).get("eval_loss")
        row["train_loss"] = train_metrics.get("train", {}).get("train_loss")
    return row


## 5. Ensure Data Files

`data/*.jsonl` files are intentionally gitignored, so a fresh Colab pull will not include them. This cell mounts Google Drive, copies files from `My Drive/RL/Data`, and only regenerates GSM8K if the clean files are still missing.

In [ ]:
import shutil
import zipfile

Path("data").mkdir(parents=True, exist_ok=True)


def mount_drive_if_needed():
    if not IN_COLAB:
        return

    drive_root = Path("/content/drive/MyDrive")
    if drive_root.exists():
        print("Google Drive already mounted.")
        return

    print("Mounting Google Drive...")
    from google.colab import drive

    drive.mount("/content/drive")


def copy_if_exists(source_path, target_path):
    source = Path(source_path)
    target = Path(target_path)
    if not source.exists():
        return False

    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists():
        print(f"Already exists: {target}")
    else:
        shutil.copy2(source, target)
        print(f"Copied {source} -> {target}")
    return True


def copy_drive_data_files():
    mount_drive_if_needed()

    drive_dir = Path(DRIVE_DATA_DIR)
    if not drive_dir.exists():
        print(f"Drive data directory not found: {drive_dir}")
        return

    print(f"Using Drive data directory: {drive_dir}")
    expected_files = [
        "gsm8k_clean_train.jsonl",
        "gsm8k_clean_test.jsonl",
        "gsm8k_teacher_preview_5000.jsonl",
        "sft_strategy_train.jsonl",
        "sft_strategy_val.jsonl",
    ]
    for filename in expected_files:
        copy_if_exists(drive_dir / filename, Path("data") / filename)


def ensure_clean_gsm8k_files():
    required = [Path(EVAL_INPUT_PATH), Path("data/gsm8k_clean_train.jsonl")]
    if all(path.exists() for path in required):
        print("Clean GSM8K files already exist.")
        return

    print("Missing clean GSM8K files after Drive copy. Running scripts/prepare_gsm8k.py first...")
    run_command(["python", "-B", "scripts/prepare_gsm8k.py"])


def find_teacher_candidate(target_path):
    target = Path(target_path)
    drive_dir = Path(DRIVE_DATA_DIR)
    candidates = [
        target,
        Path.cwd() / target.name,
        Path("/content") / target.name,
        drive_dir / target.name,
        Path("/content/drive/MyDrive") / target.name,
        Path("/content/drive/MyDrive/strategy-distill-rl") / target.name,
        Path("/content/drive/MyDrive/RL/Data") / target.name,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None



def find_adapter_source_dir(extract_dir: Path) -> Path:
    candidates = [extract_dir]
    candidates.extend(path for path in extract_dir.rglob("*") if path.is_dir())
    for candidate in candidates:
        if (candidate / "adapter_config.json").exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find adapter_config.json inside extracted checkpoint: {extract_dir}"
    )


def restore_drive_checkpoint():
    if not RUN_RESTORE_DRIVE_CHECKPOINT:
        return

    mount_drive_if_needed()
    archive = Path(DRIVE_CHECKPOINT_ARCHIVE)
    target_dir = CHECKPOINTS_DIR / DRIVE_CHECKPOINT_NAME

    if (target_dir / "adapter_config.json").exists():
        print(f"Checkpoint already restored: {target_dir}")
        return

    if not archive.exists():
        raise FileNotFoundError(
            f"Missing Drive checkpoint archive: {archive}. "
            "Update DRIVE_CHECKPOINT_ARCHIVE in the config cell."
        )

    extract_dir = Path("/tmp/strategy_distill_checkpoint_extract") / DRIVE_CHECKPOINT_NAME
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)

    print(f"Extracting checkpoint archive: {archive}")
    with zipfile.ZipFile(archive, "r") as zf:
        zf.extractall(extract_dir)

    source_dir = find_adapter_source_dir(extract_dir)
    target_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source_dir, target_dir, dirs_exist_ok=True)
    print(f"Restored checkpoint: {source_dir} -> {target_dir}")


def ensure_teacher_file():
    target = Path(TEACHER_PATH)
    if target.exists():
        print(f"Teacher file already exists: {target}")
        return

    candidate = find_teacher_candidate(target)
    if candidate is not None:
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(candidate, target)
        print(f"Copied teacher file from {candidate} -> {target}")
        return

    if IN_COLAB:
        print(f"Missing {target}. Please upload your validated teacher JSONL now.")
        print("Expected file name is usually gsm8k_teacher_preview_5000.jsonl.")
        from google.colab import files

        uploaded = files.upload()
        if not uploaded:
            raise FileNotFoundError(f"No file uploaded for {target}")

        uploaded_name = next(iter(uploaded))
        target.parent.mkdir(parents=True, exist_ok=True)
        with target.open("wb") as f:
            f.write(uploaded[uploaded_name])
        print(f"Uploaded {uploaded_name} -> {target}")
        return

    raise FileNotFoundError(
        f"Missing {target}. Copy your validated teacher JSONL to {target}, "
        "or set TEACHER_PATH to its real local path."
    )


copy_drive_data_files()
ensure_clean_gsm8k_files()
restore_drive_checkpoint()
if RUN_VALIDATE_TEACHER or RUN_BUILD_SFT or RUN_TRAINING:
    ensure_teacher_file()


## 6. Validate Teacher Data and Build SFT Files

The teacher file should already contain only usable records. Validation is still the gate before student training.

In [ ]:
if RUN_VALIDATE_TEACHER:
    run_command([
        "python", "-B", "scripts/validate_teacher_dataset.py",
        "--path", TEACHER_PATH,
    ])

if RUN_BUILD_SFT:
    run_command([
        "python", "-B", "scripts/build_sft_dataset.py",
        "--teacher-path", TEACHER_PATH,
        "--train-output", SFT_TRAIN_PATH,
        "--val-output", SFT_VAL_PATH,
        "--train-size", str(SFT_TRAIN_SIZE),
    ])


## 7. Zero-Shot Baseline

Strict `accuracy` requires a valid `<final>` block. `loose_math_accuracy` is diagnostic only: it extracts a best-effort numeric answer from raw zero-shot text so we can tell math ability apart from format following. `usable_rate` remains the main pipeline metric.

In [ ]:
baseline_output = RUNS_DIR / "zero_shot_outputs.jsonl"
baseline_metrics = RUNS_DIR / "zero_shot_metrics.json"

if RUN_ZERO_SHOT:
    run_command([
        "python", "-B", "scripts/evaluate_student.py",
        "--model-name", MODEL_NAME,
        "--input-path", EVAL_INPUT_PATH,
        "--output-path", baseline_output,
        "--metrics-path", baseline_metrics,
        "--num-samples", str(ZERO_SHOT_NUM_SAMPLES),
        "--batch-size", str(EVAL_BATCH_SIZE),
        "--max-new-tokens", str(EVAL_MAX_NEW_TOKENS),
    ])

if baseline_metrics.exists():
    display(pd.DataFrame([metric_row("zero_shot", baseline_metrics)]))

## 8. Inspect Zero-Shot Outputs

Look at a few failed or invalid outputs before training. If zero-shot already follows the format well, keep SFT lighter.

In [ ]:
if baseline_output.exists():
    rows = read_jsonl(baseline_output)
    failed = [row for row in rows if not row.get("is_usable")]
    print(f"total={len(rows)} failed_or_unusable={len(failed)}")
    for row in failed[:3]:
        print("=" * 80)
        print("id:", row["id"])
        print("question:", row["question"])
        print("ground_truth:", row["ground_truth"])
        print("model_answer:", row["model_answer"])
        print("loose_model_answer:", row.get("loose_model_answer"))
        print("is_correct:", row["is_correct"], "is_loose_correct:", row.get("is_loose_correct"), "is_format_valid:", row["is_format_valid"])
        print("format_checks:", row["format_checks"])
        print("raw_model_output:")
        print((row.get("raw_model_output") or "")[:1600])


## 9. Train LoRA SFT Configs

Set `RUN_TRAINING = True` in the config cell when ready. Start with `smoke_r8_a16_300`; only run the larger configs after the smoke run produces sane outputs.

In [ ]:
trained_configs = []

if RUN_TRAINING:
    for cfg in TRAIN_CONFIGS:
        if cfg["name"] not in ACTIVE_TRAIN_CONFIG_NAMES:
            continue
        output_dir = CHECKPOINTS_DIR / cfg["name"]
        run_command([
            "python", "-B", "scripts/train_sft_lora.py",
            "--model-name", MODEL_NAME,
            "--train-path", SFT_TRAIN_PATH,
            "--val-path", SFT_VAL_PATH,
            "--output-dir", output_dir,
            "--max-train-samples", str(cfg["max_train_samples"]),
            "--max-val-samples", str(cfg["max_val_samples"]),
            "--num-train-epochs", str(cfg["epochs"]),
            "--learning-rate", str(cfg["lr"]),
            "--lora-r", str(cfg["lora_r"]),
            "--lora-alpha", str(cfg["lora_alpha"]),
            "--lora-dropout", str(cfg["lora_dropout"]),
            "--gradient-accumulation-steps", str(cfg["grad_accum"]),
        ])
        trained_configs.append({**cfg, "output_dir": output_dir})
else:
    for cfg in TRAIN_CONFIGS:
        if cfg["name"] not in ACTIVE_TRAIN_CONFIG_NAMES:
            continue
        output_dir = CHECKPOINTS_DIR / cfg["name"]
        if output_dir.exists():
            trained_configs.append({**cfg, "output_dir": output_dir})

print("Adapters available for eval:")
for cfg in trained_configs:
    print(cfg["name"], "->", cfg["output_dir"])


## 10. Evaluate Trained Adapters

This uses the exact same evaluator as zero-shot, so the comparison is apples-to-apples.

In [ ]:
adapter_eval_rows = []

if RUN_ADAPTER_EVAL:
    for cfg in trained_configs:
        name = cfg["name"]
        adapter_path = cfg["output_dir"]
        output_path = RUNS_DIR / f"{name}_outputs.jsonl"
        metrics_path = RUNS_DIR / f"{name}_metrics.json"
        train_metrics_path = adapter_path / "train_metrics.json"

        run_command([
            "python", "-B", "scripts/evaluate_student.py",
            "--model-name", MODEL_NAME,
            "--adapter-path", adapter_path,
            "--input-path", EVAL_INPUT_PATH,
            "--output-path", output_path,
            "--metrics-path", metrics_path,
            "--num-samples", str(FINAL_EVAL_NUM_SAMPLES),
            "--batch-size", str(EVAL_BATCH_SIZE),
            "--max-new-tokens", str(EVAL_MAX_NEW_TOKENS),
        ])
        adapter_eval_rows.append(metric_row(name, metrics_path, train_metrics_path))

adapter_eval_rows

## 11. Compare Metrics

Prioritize `usable_rate` first, then strict `accuracy`, then `format_valid_rate`. Use `loose_math_accuracy` only as a diagnostic for zero-shot/base-model math ability when the model ignores the required XML format.

In [ ]:
comparison_rows = []
if baseline_metrics.exists():
    comparison_rows.append(metric_row("zero_shot", baseline_metrics))

for cfg in TRAIN_CONFIGS:
    name = cfg["name"]
    metrics_path = RUNS_DIR / f"{name}_metrics.json"
    train_metrics_path = CHECKPOINTS_DIR / name / "train_metrics.json"
    if metrics_path.exists():
        comparison_rows.append(metric_row(name, metrics_path, train_metrics_path))

comparison = pd.DataFrame(comparison_rows)
if not comparison.empty:
    display(
        comparison.sort_values(
            by=["usable_rate", "accuracy", "format_valid_rate"],
            ascending=False,
        )
    )
    comparison.to_csv(RUNS_DIR / "comparison.csv", index=False)
    print(f"Saved comparison to {RUNS_DIR / 'comparison.csv'}")
else:
    print("No metrics found yet.")

## 12. Output Format Audit

Audit one eval output JSONL file, count strict-format failures, answer failures, unusable samples, failed checklist items, and print up to `AUDIT_SAMPLE_COUNT` examples for manual inspection. Error samples are shown first.

In [ ]:
from collections import Counter


def choose_audit_output_path():
    if AUDIT_RUN_NAME != "auto":
        if AUDIT_RUN_NAME == "zero_shot":
            return "zero_shot", baseline_output
        return AUDIT_RUN_NAME, RUNS_DIR / f"{AUDIT_RUN_NAME}_outputs.jsonl"

    for name in ACTIVE_TRAIN_CONFIG_NAMES:
        candidate = RUNS_DIR / f"{name}_outputs.jsonl"
        if candidate.exists():
            return name, candidate

    if baseline_output.exists():
        return "zero_shot", baseline_output

    return None, None


def audit_eval_records(records):
    total = len(records)
    format_invalid = [row for row in records if not row.get("is_format_valid")]
    strict_wrong = [row for row in records if not row.get("is_correct")]
    loose_wrong = [row for row in records if not row.get("is_loose_correct")]
    unusable = [row for row in records if not row.get("is_usable")]

    failed_checks = Counter()
    for row in records:
        for check_name, passed in row.get("format_checks", {}).items():
            if not passed:
                failed_checks[check_name] += 1

    summary = {
        "total": total,
        "strict_correct": total - len(strict_wrong),
        "strict_wrong": len(strict_wrong),
        "loose_correct": total - len(loose_wrong),
        "loose_wrong": len(loose_wrong),
        "format_valid": total - len(format_invalid),
        "format_invalid": len(format_invalid),
        "usable": total - len(unusable),
        "unusable": len(unusable),
        "strict_accuracy": (total - len(strict_wrong)) / total if total else 0.0,
        "loose_math_accuracy": (total - len(loose_wrong)) / total if total else 0.0,
        "format_valid_rate": (total - len(format_invalid)) / total if total else 0.0,
        "usable_rate": (total - len(unusable)) / total if total else 0.0,
    }
    return summary, failed_checks, unusable


def print_audit_samples(records, max_samples=50):
    # Show problematic rows first, then fill with valid rows for a broad spot check.
    sorted_rows = sorted(
        records,
        key=lambda row: (
            row.get("is_usable", 0),
            row.get("is_format_valid", 0),
            row.get("is_correct", 0),
            row.get("id", 0),
        ),
    )

    for idx, row in enumerate(sorted_rows[:max_samples], start=1):
        print("=" * 100)
        print(f"sample #{idx} | id={row.get('id')}")
        print("question:", row.get("question"))
        print("ground_truth:", row.get("ground_truth"))
        print("model_answer:", row.get("model_answer"))
        print("loose_model_answer:", row.get("loose_model_answer"))
        print(
            "is_correct:", row.get("is_correct"),
            "is_loose_correct:", row.get("is_loose_correct"),
            "is_format_valid:", row.get("is_format_valid"),
            "is_usable:", row.get("is_usable"),
        )
        print("format_checks:", row.get("format_checks"))
        print("model_output:")
        print(row.get("model_output"))
        print("raw_model_output preview:")
        print((row.get("raw_model_output") or "")[:1800])


audit_run_name, audit_path = choose_audit_output_path()
if audit_path is None or not audit_path.exists():
    print("No eval output file found yet. Run zero-shot or adapter eval first.")
else:
    audit_records = read_jsonl(audit_path)
    summary, failed_checks, unusable_records = audit_eval_records(audit_records)

    print(f"Auditing run: {audit_run_name}")
    print(f"Audit file: {audit_path}")
    display(pd.DataFrame([summary]))

    failed_check_df = pd.DataFrame(
        [{"check": name, "failed_count": count} for name, count in failed_checks.most_common()]
    )
    if not failed_check_df.empty:
        display(failed_check_df)
    else:
        print("No failed format checks.")

    print(
        f"Total={summary['total']} | "
        f"format_invalid={summary['format_invalid']} | "
        f"strict_wrong={summary['strict_wrong']} | "
        f"loose_wrong={summary['loose_wrong']} | "
        f"unusable={summary['unusable']}"
    )
    print(f"\nPrinting up to {AUDIT_SAMPLE_COUNT} samples, with errors first.")
    print_audit_samples(audit_records, max_samples=AUDIT_SAMPLE_COUNT)


## 13. Save SFT Checkpoint to Drive

Archive the active SFT adapter after training so it can be reused for rollout generation and DAPO without retraining.


In [ ]:
if IN_COLAB and RUN_SAVE_SFT_CHECKPOINT_TO_DRIVE:
    import shutil
    from datetime import datetime
    from pathlib import Path

    mount_drive_if_needed()
    drive_checkpoint_dir = Path(DRIVE_CHECKPOINT_DIR)
    drive_checkpoint_dir.mkdir(parents=True, exist_ok=True)

    active_name = ACTIVE_TRAIN_CONFIG_NAMES[0]
    active_checkpoint_dir = CHECKPOINTS_DIR / active_name
    if not active_checkpoint_dir.exists():
        raise FileNotFoundError(f"Missing active checkpoint directory: {active_checkpoint_dir}")

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    archive_base = Path("/tmp") / f"{active_name}_{timestamp}"
    archive_path = shutil.make_archive(str(archive_base), "zip", active_checkpoint_dir)
    drive_archive_path = drive_checkpoint_dir / Path(archive_path).name
    shutil.copy2(archive_path, drive_archive_path)

    print("Saved SFT checkpoint archive to:")
    print(drive_archive_path)
    print("Archive size MB:", drive_archive_path.stat().st_size / (1024 * 1024))
else:
    print("Not in Colab or RUN_SAVE_SFT_CHECKPOINT_TO_DRIVE=False; skipping SFT checkpoint archive.")
